In [ ]:
import kagglehub
path = kagglehub.dataset_download("manjilkarki/deepfake-and-real-images")

Using Colab cache for faster access to the 'deepfake-and-real-images' dataset.


In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
import matplotlib.pyplot as plt
print("Complete")

Complete


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_dir = path + "/Dataset/Train"
test_dir  = path + "/Dataset/Test"


datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)


train_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(256, 256),
    batch_size=32,
    class_mode="binary",
    subset="training"
)

val_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(256, 256),
    batch_size=32,
    class_mode="binary",
    subset="validation"
)

test_generator = datagen.flow_from_directory(
    test_dir,
    target_size=(256, 256),
    batch_size=32,
    class_mode="binary"
)

Found 112002 images belonging to 2 classes.
Found 28000 images belonging to 2 classes.
Found 10905 images belonging to 2 classes.


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input

model = Sequential([
    Conv2D(8, (3,3), activation='relu', input_shape=(256,256,3)),
    MaxPooling2D(2,2),
    Conv2D(16, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(100, activation='relu'),
    Dropout(0.25),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.summary()
print("Complete")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 254, 254, 8)    │           224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 127, 127, 8)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 125, 125, 16)   │         1,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 62, 62, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 60, 60, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 30, 30, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 28, 28, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 100)            │     1,254,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           101 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,279,129 (4.88 MB)

 Trainable params: 1,279,129 (4.88 MB)

 Non-trainable params: 0 (0.00 B)

Complete


In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10
)
model.save("deepfake_model.h5")
print("Complete")

Epoch 1/10
3501/3501 ━━━━━━━━━━━━━━━━━━━━ 0s 194ms/step - accuracy: 0.7887 - loss: 0.4245

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


3501/3501 ━━━━━━━━━━━━━━━━━━━━ 854s 242ms/step - accuracy: 0.7887 - loss: 0.4245 - val_accuracy: 0.9209 - val_loss: 0.1904
Epoch 2/10
3501/3501 ━━━━━━━━━━━━━━━━━━━━ 269s 77ms/step - accuracy: 0.9304 - loss: 0.1697 - val_accuracy: 0.9236 - val_loss: 0.1798
Epoch 3/10
3501/3501 ━━━━━━━━━━━━━━━━━━━━ 269s 77ms/step - accuracy: 0.9475 - loss: 0.1288 - val_accuracy: 0.9436 - val_loss: 0.1377
Epoch 4/10
3501/3501 ━━━━━━━━━━━━━━━━━━━━ 264s 75ms/step - accuracy: 0.9579 - loss: 0.1046 - val_accuracy: 0.9461 - val_loss: 0.1365
Epoch 5/10
3501/3501 ━━━━━━━━━━━━━━━━━━━━ 272s 78ms/step - accuracy: 0.9652 - loss: 0.0892 - val_accuracy: 0.9497 - val_loss: 0.1310
Epoch 6/10
3501/3501 ━━━━━━━━━━━━━━━━━━━━ 265s 76ms/step - accuracy: 0.9703 - loss: 0.0738 - val_accuracy: 0.9554 - val_loss: 0.1158
Epoch 7/10
3501/3501 ━━━━━━━━━━━━━━━━━━━━ 283s 81ms/step - accuracy: 0.9742 - loss: 0.0642 - val_accuracy: 0.9581 - val_loss: 0.1177
Epoch 8/10
3501/3501 ━━━━━━━━━━━━━━━━━━━━ 269s 77ms/step - accuracy: 0.9779 - l

Complete


In [ ]:
model.save('my_model.keras')

In [ ]:
model.save("deepfake_model2.h5")

In [ ]:
loss, accuracy = model.evaluate(test_generator)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


341/341 ━━━━━━━━━━━━━━━━━━━━ 76s 222ms/step - accuracy: 0.8728 - loss: 0.4349
Test Loss: 0.4200
Test Accuracy: 0.8740
